In [0]:
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .csv("/Volumes/projects/project_workouts/retail_sales1/ecommerce_sales.csv")

display(df_raw)

In [0]:
print("Total rows:", df_raw.count())

In [0]:
print(df_raw.columns)

In [0]:
df_raw.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
]).show()

In [0]:
print(
    "Unique orders:",
    df_raw.select("order_id").distinct().count()
)

In [0]:
from pyspark.sql.functions import trim

df_clean = (
    df_raw
    .dropDuplicates(["order_id"])
    .dropna(
        subset=[
            "order_id",
            "customer_id",
            "product"
        ]
    )
    .withColumn("product", trim(col("product")))
    .withColumn("category", trim(col("category")))
    .withColumn("city", trim(col("city")))
    .withColumn(
        "payment_method",
        trim(col("payment_method"))
    )
)

display(df_clean)

In [0]:
print("Clean records:", df_clean.count())

In [0]:
from pyspark.sql.functions import round

df_clean = df_clean.withColumn(
    "calculated_amount",
    round(
        col("quantity") *
        col("unit_price"),
        2
    )
)

display(df_clean)

In [0]:
from pyspark.sql.functions import count, sum, avg

df_gold = (
    df_clean
    .groupBy("product", "category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(
            sum("calculated_amount"),
            2
        ).alias("total_sales"),
        round(
            avg("calculated_amount"),
            2
        ).alias("average_order_value")
    )
    .orderBy(
        col("total_sales").desc()
    )
)

display(df_gold)

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projects.project2.ecommerce_sales_silver")

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projects.project2.ecommerce_sales_gold")

In [0]:
%sql
SELECT
    *
FROM projects.project2.ecommerce_sales_gold
ORDER BY total_sales DESC;

In [0]:
%sql
SELECT
    city,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(calculated_amount), 2) AS total_sales
FROM projects.project2.ecommerce_sales_silver
GROUP BY city
ORDER BY total_sales DESC;

In [0]:
%sql
SELECT
    payment_method,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(calculated_amount), 2) AS total_sales
FROM projects.project2.ecommerce_sales_silver
GROUP BY payment_method
ORDER BY total_sales DESC;

In [0]:
%sql
select
    category,
    count(order_id) as total_orders,
    sum(quantity) as total_quantity,
    round(sum(calculated_amount),2) as total_sales
    from projects.project2.ecommerce_sales_silver
    group by category
    order by total_sales desc

                 E-Commerce CSV
                       │
                       ▼
                 RAW DATA
                       │
                       ▼
               PySpark Validation
                       │
                       ▼
                CLEAN DATA
                       │
                       ▼
               Transformations
                       │
                       ▼
                Delta Lake
                       │
              ┌────────┴────────┐
              ▼                 ▼
          Silver Layer      Gold Layer
              │                 │
              └────────┬────────┘
                       ▼
                  SQL Analytics